In [ ]:
! pip install synthid-text[notebook]

from collections.abc import Sequence
import enum
import gc

import datasets
import huggingface_hub
from synthid_text import detector_mean
from synthid_text import logits_processing
from synthid_text import synthid_mixin
#from synthid_text import detector_bayesian
import tensorflow as tf
import torch
import tqdm
import transformers
import immutabledict

In [ ]:
ATTACK_ENABLED = True

from collections.abc import Sequence
import enum
import gc
import os

import datasets
import huggingface_hub
from synthid_text import detector_mean
from synthid_text import logits_processing
from synthid_text import synthid_mixin
from synthid_text import detector_bayesian
import tensorflow as tf
import torch
import tqdm
import transformers
import immutabledict


PREFERRED_CUDA_DEVICE = os.environ.get("DEVICE_CUDA", "cuda")

import json
import tensorflow_datasets as tfds
import datasets

ds = datasets.load_dataset("Pavithree/eli5")
id_to_prompt = {}
# Access the 'train' split of the dataset and iterate over it
for x in iter(ds['train'].to_iterable_dataset()):
    id_to_prompt[x['q_id']] = x['title']

full_data = []
with open('human_eval.jsonl') as f:
    for json_str in f:
        x = json.loads(json_str)
        # Check if the q_id exists in id_to_prompt before accessing it
        if x['q_id'] in id_to_prompt:
            x['question'] = id_to_prompt[x['q_id']]
            full_data.append(x)

def _resolve_device(preferred_cuda_device: str) -> torch.device:
  """Return a valid CUDA device, or CPU as fallback."""
  if preferred_cuda_device.lower() == "cpu" or not torch.cuda.is_available():
    return torch.device("cpu")

  try:
    if preferred_cuda_device == "cuda":
      requested_index = 0
    elif preferred_cuda_device.startswith("cuda:"):
      requested_index = int(preferred_cuda_device.split(":", 1)[1])
    else:
      requested_index = int(preferred_cuda_device)
  except ValueError:
    requested_index = 0

  visible_gpu_count = torch.cuda.device_count()
  if requested_index < 0 or requested_index >= visible_gpu_count:
    requested_index = 0

  device = torch.device(f"cuda:{requested_index}")
  try:
    torch.empty(1, device=device)
  except RuntimeError:
    return torch.device("cpu")

  return device




In [ ]:
class ModelName(enum.Enum):
  GPT2 = 'gpt2'
  GEMMA_2B = 'google/gemma-2b-it'
  GEMMA_7B = 'google/gemma-7b-it'


model_name = 'google/gemma-7b-it' # @param ['gpt2', 'google/gemma-2b-it', 'google/gemma-7b-it']
MODEL_NAME = ModelName(model_name)


if MODEL_NAME is not ModelName.GPT2:
  huggingface_hub.notebook_login()

In [10]:


# Tokenizer
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME.value)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# SyntHID config
DEVICE = _resolve_device(PREFERRED_CUDA_DEVICE)
DEVICE_MAP = str(DEVICE)
DEVICE


DEFAULT_WATERMARKING_CONFIG = immutabledict.immutabledict({
    "ngram_len": 5,  # This corresponds to H=4 context window size in the paper.
    "keys": [
        654,
        400,
        836,
        123,
        340,
        443,
        597,
        160,
        57,
        29,
        590,
        639,
        13,
        715,
        468,
        990,
        966,
        226,
        324,
        585,
        118,
        504,
        421,
        521,
        129,
        669,
        732,
        225,
        90,
        960,
        654,
        400,
        836,
        123,
        340,
        443,
        597,
        160,
        57,
        29,
        590,
        639,
        13,
        715,
        468,
        990,
        966,
        226,
        324,
        585,
        118,
        504,
        421,
        521,
        129,
        669,
        732,
        225,
        90,
        960,
        654,
        400,
        836,
        123,
        340,
        443,
        597,
        160,
        57,
        29,
        590,
        639,
        13,
        715,
        468,
        990,
        966,
        226,
        324,
        585,
        118,
        504,
        421,
        521,
        129,
        669,
        732,
        225,
        90,
        960,
    ],
    "sampling_table_size": 2**16,
    "context_history_size": 1024,
    'sampling_table_seed':0,
    "device": DEVICE,
})
CONFIG = DEFAULT_WATERMARKING_CONFIG

BATCH_SIZE = 8
NUM_BATCHES = 320
OUTPUTS_LEN = 1024
TOP_K = 40
TOP_P = 0.99
NUM_NEGATIVES = 10000
POS_BATCH_SIZE = 32
NUM_POS_BATCHES = 313
NEG_BATCH_SIZE = 32
# Truncate outputs to this length for training.
POS_TRUNCATION_LENGTH = 200
NEG_TRUNCATION_LENGTH = 200
# Pad trucated outputs to this length for equal shape across all batches.
MAX_PADDED_LENGTH = 1000
TEMPERATURE = 1.0

# Utility function
def load_model(
    model_name: ModelName,
    expected_device: torch.device,
    enable_watermarking: bool = False,
) -> transformers.PreTrainedModel:
  if model_name == ModelName.GPT2:
    model_cls = (
        synthid_mixin.SynthIDGPT2LMHeadModel
        if enable_watermarking
        else transformers.GPT2LMHeadModel
    )
    model = model_cls.from_pretrained(model_name.value, device_map=DEVICE_MAP)
    model.generation_config.pad_token_id = model.generation_config.eos_token_id
  else:
    model_cls = (
        synthid_mixin.SynthIDGemmaForCausalLM
        if enable_watermarking
        else transformers.GemmaForCausalLM
    )
    model = model_cls.from_pretrained(
        model_name.value,
        device_map=DEVICE_MAP,
        torch_dtype=(torch.bfloat16 if DEVICE.type == "cuda" else torch.float32),
    )

  if str(model.device) != str(expected_device):
    raise ValueError('Model device not as expected.')

  return model


def _compute_perplexity(
    outputs: torch.LongTensor,
    scores: torch.FloatTensor,
    eos_token_mask: torch.LongTensor,
    watermarked: bool = False,
) -> float:
  """Compute perplexity given the model outputs and the logits."""
  len_offset = len(scores)
  if watermarked:
    nll_scores = scores
  else:
    nll_scores = [
        torch.gather(
            -torch.log(torch.nn.Softmax(dim=1)(sc)),
            1,
            outputs[:, -len_offset + idx, None],
        )
        for idx, sc in enumerate(scores)
    ]
  nll_sum = torch.nan_to_num(
      torch.squeeze(torch.stack(nll_scores, dim=1), dim=2)
      * eos_token_mask.long(),
      posinf=0,
  )
  nll_sum = nll_sum.sum(dim=1)
  nll_mean = nll_sum / eos_token_mask.sum(dim=1)
  return nll_mean.sum(dim=0)


def _process_raw_prompt(prompt: Sequence[str]) -> str:
  """Add chat template to the raw prompt."""
  if MODEL_NAME == ModelName.GPT2:
    return prompt.decode().strip('"')
  else:
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt.decode().strip('"')}],
        tokenize=False,
        add_generation_prompt=True,
    )

# @title Generate model responses and compute g-values

def generate_responses(example_inputs, enable_watermarking):
  inputs = tokenizer(
      example_inputs,
      return_tensors='pt',
      padding=True,
  ).to(DEVICE)

  # @title Watermarked output preparation for detector training
  gc.collect()
  if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

  model = load_model(
      MODEL_NAME,
      expected_device=DEVICE,
      enable_watermarking=enable_watermarking,
  )
  _, inputs_len = inputs['input_ids'].shape

  outputs = model.generate(
      **inputs,
      do_sample=True,
      max_length=inputs_len + OUTPUTS_LEN,
      temperature=TEMPERATURE,
      top_k=TOP_K,
      top_p=TOP_P,
  )

  outputs = outputs[:, inputs_len:]

  # eos mask is computed, skip first ngram_len - 1 tokens
  # eos_mask will be of shape [batch_size, output_len]
  eos_token_mask = logits_processor.compute_eos_token_mask(
      input_ids=outputs,
      eos_token_id=tokenizer.eos_token_id,
  )[:, CONFIG['ngram_len'] - 1 :]

  # context repetition mask is computed
  context_repetition_mask = logits_processor.compute_context_repetition_mask(
      input_ids=outputs,
  )
  # context repitition mask shape [batch_size, output_len - (ngram_len - 1)]

  combined_mask = context_repetition_mask * eos_token_mask

  g_values = logits_processor.compute_g_values(
      input_ids=outputs,
  )
  # g values shape [batch_size, output_len - (ngram_len - 1), depth]

  return g_values, combined_mask

import numpy as np
from sklearn.metrics import roc_curve, auc
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

def find_best_threshold_at_fpr(y_true, y_scores, target_fpr=0.01):

    # Calculate ROC curve
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)

    # Find the threshold where FPR is closest to target_fpr
    idx = np.argmin(np.abs(fpr - target_fpr))

    best_threshold = thresholds[idx]
    best_tpr = tpr[idx]
    actual_fpr = fpr[idx]

    return best_threshold, best_tpr, actual_fpr




In [ ]:
model = load_model(MODEL_NAME, expected_device=DEVICE, enable_watermarking=False)
logits_processor = logits_processing.SynthIDLogitsProcessor(
    **CONFIG, top_k=TOP_K, temperature=TEMPERATURE)

In [ ]:
# Attack
from collections import Counter
import numpy as np


def tournament_layer(tokens):
    count = Counter(tokens)
    unique_tokens = set(tokens)
    g_values = {x: -count[x] for x in unique_tokens}

    winners = []
    for i in range(0, len(tokens), 2):
        t1, t2 = tokens[i], tokens[i+1]
        if g_values[t1] > g_values[t2]:
            winners.append(t1)
        elif g_values[t2] > g_values[t1]:
            winners.append(t2)
        else:  # tie → random choice
            winners.append(np.random.choice([t1, t2]))
    return winners

def tournament_model(tokens, num_layers):
    #print(tokens)
    for _ in range(num_layers):
        tokens = tournament_layer(tokens)
    return tokens


def sample_candidates_with_watermark(
    input_ids,
    attention_mask,
    watermark_processor,
    num_candidates,
):
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )
        next_token_logits = outputs.logits[:, -1, :]

    watermarked_scores, _, _ = watermark_processor.watermarked_call(
        input_ids=input_ids,
        scores=next_token_logits,
    )

    probs = torch.softmax(watermarked_scores[0] / 1.0, dim=-1)
    sampled_token_ids = torch.multinomial(
        probs,
        num_samples=num_candidates,
        replacement=True,
    )
    return sampled_token_ids.tolist()


def format_generation_prompt(prompt: str) -> str:
    """Format prompt for current model family before generation."""
    if MODEL_NAME == ModelName.GPT2:
        return prompt
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )


from scipy.special import softmax
import tqdm
from synthid_text import logits_processing
import torch

num_layer_att = 10
num_detect = 0
tau = 0.5106
GENERATION_TEMPERATURE = 1.0
GENERATION_TOP_K = 40
GENERATION_MAX_NEW_TOKENS = 100

# Ensure model and config are loaded (assuming they are from previous cells)

# We iterate through a subset for demonstration to save time, or use the full range as requested
for i in range(100): # Reduced range for quick testing, change back to 1000 for full run
    prompt = full_data[i]
    generation_prompt = format_generation_prompt(prompt)
    prompt_ids = tokenizer(
        generation_prompt,
        return_tensors='pt',
        add_special_tokens=False,
    )['input_ids'].to(DEVICE)
    attention_mask = torch.ones_like(prompt_ids)
    generated_token_ids = []

    # Keep one watermark processor per generated sample to preserve state
    # across token steps. Re-creating this per step breaks watermarking.
    generation_logits_processor = logits_processing.SynthIDLogitsProcessor(
        **CONFIG,
        top_k=GENERATION_TOP_K,
        temperature=float(GENERATION_TEMPERATURE),
    )

    for _ in tqdm.tqdm(range(GENERATION_MAX_NEW_TOKENS)):
        new_tokens = sample_candidates_with_watermark(
            input_ids=prompt_ids,
            attention_mask=attention_mask,
            watermark_processor=generation_logits_processor,
            num_candidates=2 ** num_layer_att,
        )

        if ATTACK_ENABLED:
            winner_token_id = tournament_model(new_tokens, num_layer_att)
        else:
            winner_token_id = new_tokens[0]  # attack disabled for debugging
        if isinstance(winner_token_id, list):
            winner_token_id = winner_token_id[0]

        if winner_token_id == tokenizer.eos_token_id:
            continue

        winner_token = torch.tensor([[winner_token_id]], dtype=torch.long, device=DEVICE)
        prompt_ids = torch.concat((prompt_ids, winner_token), dim=1)
        attention_mask = torch.concat(
            (
                attention_mask,
                torch.ones((1, 1), dtype=attention_mask.dtype, device=DEVICE),
            ),
            dim=1,
        )
        generated_token_ids.append(winner_token_id)

    if len(generated_token_ids) < CONFIG['ngram_len']:
        print(
            f"Example {i} skipped: only {len(generated_token_ids)} generated tokens."
        )
        continue

    output_ids = torch.tensor(
        [generated_token_ids],
        dtype=torch.long,
        device=DEVICE,
    )

    # Evaluation code needs to handle the final output
    eos_token_mask = logits_processor.compute_eos_token_mask(
        input_ids=output_ids,
        eos_token_id=tokenizer.eos_token_id,
    )[:, CONFIG['ngram_len'] - 1 :]

    context_repetition_mask = logits_processor.compute_context_repetition_mask(
        input_ids=output_ids
    )

    combined_mask = context_repetition_mask * eos_token_mask

    # g_values computation requires input_ids
    g_values = logits_processor.compute_g_values(
        input_ids=output_ids
    )

    # Score
    score = detector_mean.mean_score(g_values.cpu().numpy(), combined_mask.cpu().numpy())
    score = float(np.asarray(score).reshape(-1)[0])

    # Assuming 'tau' is defined somewhere or we pick a threshold.
    # If tau is not defined, we can print the score.
    if score > tau:
       num_detect += 1
    print(f"Example {i} Score: {score}")

print(num_detect)
